# Notebook 2: Engenharia e Seleção de Features

**Unidade Curricular:** Dados e Aprendizagem Automática (DAA) — 2026/2027

Este notebook parte dos dados limpos do Notebook 1 e prepara-os para a modelação:

1. **Codificação** das variáveis categóricas em números (requisito dos modelos);
2. **Seleção** de features, removendo colunas sem informação ou redundantes;
3. **Criação** de novas features a partir do conhecimento obtido na análise exploratória.

## 1. Configuração e Carregamento dos Dados

Carregamos os dados produzidos pelo Notebook 1, com o `PID` como índice. Na leitura, apenas as células vazias são tratadas como valores em falta (`keep_default_na=False`), para que o pandas não converta textos como "NA" ou "None" em `NaN` por engano.

In [6]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 300)
plt.style.use('ggplot')

DATA_DIR = '../data'
TARGET = 'AGE_RANGE'
RANDOM_STATE = 42
CODIGOS = ['Sem resposta', 'NR', 'Não aplicável']

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'), index_col='PID',
                       keep_default_na=False, na_values=[''])
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_nb01.csv'), index_col='PID',
                      keep_default_na=False, na_values=[''])

cols_num = train_df.select_dtypes(include='number').columns.drop(TARGET).tolist()
cols_cat = train_df.select_dtypes(include='object').columns.tolist()

print(f"Treino: {train_df.shape} | Teste: {test_df.shape}")
print(f"Numéricas: {len(cols_num)} | Categóricas: {len(cols_cat)}")

Treino: (10962, 261) | Teste: (3655, 260)
Numéricas: 33 | Categóricas: 227


/tmp/ipykernel_795114/3270258295.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols_cat = train_df.select_dtypes(include='object').columns.tolist()


## 2. Inventário das Escalas de Resposta

Muitas perguntas do questionário partilham a mesma escala de resposta (ex.: "Nenhuma / Alguma / Muita dificuldade"). Agrupamos as variáveis categóricas pelo seu conjunto de respostas (sem os códigos "Sem resposta" e "Não aplicável"), para definir a codificação uma vez por escala, e não uma vez por coluna.

In [7]:
escalas = {}
for c in cols_cat:
    respostas = pd.concat([train_df[c], test_df[c]]).unique()
    respostas = tuple(sorted(set(respostas) - set(CODIGOS)))
    escalas.setdefault(respostas, []).append(c)

inventario = pd.DataFrame([{'Nº colunas': len(cols), 'Nº respostas': len(resp),
                            'Respostas': ' | '.join(resp), 'Colunas (exemplo)': ', '.join(cols[:4])}
                           for resp, cols in escalas.items()])
inventario = inventario.sort_values('Nº colunas', ascending=False).reset_index(drop=True)

print(f"{len(cols_cat)} colunas categóricas -> {len(inventario)} escalas diferentes")
with pd.option_context('display.max_colwidth', None):
    display(inventario)

227 colunas categóricas -> 70 escalas diferentes


,Nº colunas,Nº respostas,Respostas,Colunas (exemplo)
0,78,2,Não | Sim,"IN1, IN6, IN7a, IN7b"
1,29,1,Valor,"IN2_ESTADO, IN3_ESTADO, IN4_ESTADO, HHNBPERS_COD_ESTADO"
2,15,4,Alguma dificuldade | Muita dificuldade | Nenhuma dificuldade | Não consegue de todo,"PL2, PL4, PL5, IN8"
3,8,4,Mais do que metade dos dias | Nunca | Quase todos os dias | Vários dias,"MH1A, MH1B, MH1C, MH1D"
4,8,5,1 a 3 vezes por semana | 4 a 6 vezes por semana | Menos de uma vez por semana | Nunca | Uma vez ou mais por dia,"DH1, DH3, DH5, DH6"
5,7,5,Alguma dificuldade | Muita dificuldade | Nenhuma dificuldade | Não aplicável (nunca tentou ou não precisa de o fazer) | Não consegue de todo,"HA1A, HA1B, HA1C, HA1D"
6,5,7,De acordo | Em desacordo | Mais ou menos de acordo | Mais ou menos em desacordo | Nem de acordo nem em desacordo | Totalmente de acordo | Totalmente em desacordo,"IN50, IN51, IN52, IN53"
7,4,3,Não houve necessidade | Precisou e não satisfez essa necessidade | Precisou e satisfez essa necessidade,"UN2A, UN2B, UN2C, UN2D"
8,3,5,5 anos ou mais | Entre 1 ano e menos de 3 anos | Entre 3 anos e menos de 5 anos | Nos últimos 12 meses | Nunca,"PA2, PA3, PA4"
9,3,3,Autóctone (PT) | Outro estado membro da UE | Outro país fora da UE,"BIRTHPLACE_COD, BIRTHPLACEFATH_COD, BIRTHPLACEMOTH_COD"


## 3. Codificação das Variáveis Categóricas

Os modelos só aceitam números, pelo que cada variável categórica é codificada consoante o seu tipo:

- **Ordinais e binárias** (ex.: "Nenhuma / Alguma / Muita dificuldade", "Não / Sim"): as respostas são substituídas pela sua posição na escala (0, 1, 2, ...), preservando a ordem;
- **Nominais** (sem ordem, ex.: região, situação profissional): *one-hot encoding*, com uma coluna binária por resposta.

Os códigos do inquérito, que não pertencem à escala, recebem valores negativos: **−1** para "Não aplicável", **−2** para "Sem resposta" e **−3** para respostas especiais fora da escala ("Não aplicável (nunca tentou ou não precisa de o fazer)", "Não consegue avaliar"). Mantém-se assim a distinção entre não aplicável e não respondido, essencial por exemplo nas perguntas de autonomia, só feitas a partir dos 55 anos.

### 3.1 Definição das Escalas

Cada escala é definida uma única vez, como uma lista de respostas por ordem crescente. Cada coluna é associada automaticamente à escala que contém todas as suas respostas.

In [8]:
def escala(respostas, especiais=None):
    """Lista ordenada de respostas -> {resposta: 0, 1, 2, ...}, mais respostas especiais."""
    mapa = {r: i for i, r in enumerate(respostas)}
    mapa.update(especiais or {})
    return mapa


CODIGOS_NUM = {'Não aplicável': -1, 'Sem resposta': -2, 'NR': -2}

ESCALAS = [
    # Binárias e variantes
    escala(['Não', 'Sim'], {'Cego (não consegue ver de todo)': 2, 'Surdo (surdez profunda)': 2,
                            'Desloca-se em cadeira de rodas': 2}),
    escala(['Não houve necessidade', 'Não', 'Sim']),
    escala(['Valor', 'Gravidez sem termo (n.a.)']),
    escala(['Homem', 'Mulher']),
    escala(['Sem relação de conjugalidade', 'Com relação de conjugalidade legal ou de facto']),
    escala(['Tempo parcial', 'Tempo completo']),
    escala(['Não utiliza', 'Utiliza']),

    # Dificuldade, limitação, dor, saúde
    escala(['Nenhuma dificuldade', 'Alguma dificuldade', 'Muita dificuldade', 'Não consegue de todo'],
           {'Não aplicável (nunca tentou ou não precisa de o fazer)': -3}),
    escala(['Sozinho, sem dificuldade', 'Sozinho, mas com dificuldade', 'Só com ajuda']),
    escala(['Muito bom', 'Bom', 'Razoável', 'Mau', 'Muito mau']),
    escala(['Nada limitado', 'Limitado mas não severamente', 'Severamente limitado']),
    escala(['Nenhuma', 'Muito ligeira', 'Ligeira', 'Moderada', 'Intensa', 'Muito intensa']),
    escala(['Nada', 'Um pouco', 'De forma moderada', 'Bastante', 'De forma extrema']),
    escala(['Nunca', 'Vários dias', 'Mais do que metade dos dias', 'Quase todos os dias']),
    escala(['Não foi necessária intervenção', 'Intervenção médica ou de enfermeiro',
            'Cuidados em hospital (com ou sem internamento)']),
    escala(['1-7', '8-14', '15-30', '31-180', '>= 181']),
    escala(['Não houve necessidade', 'Precisou e satisfez essa necessidade',
            'Precisou e não satisfez essa necessidade']),

    # Há quanto tempo (do mais recente ao "Nunca")
    escala(['Nos últimos 12 meses', 'Entre 1 ano e menos de 3 anos', 'Entre 3 anos e menos de 5 anos',
            '5 anos ou mais', 'Nunca']),
    escala(['Nos últimos 12 meses', 'Entre 1 ano e menos de 2 anos', 'Entre 2 anos e menos de 3 anos',
            'Entre 3 anos e menos de 5 anos', '5 anos ou mais', 'Nunca']),
    escala(['Nos últimos 12 meses', 'Entre 1 ano e menos de 2 anos', 'Entre 2 anos e menos de 3 anos',
            '3 anos ou mais', 'Nunca']),
    escala(['Nos últimos 12 meses', 'Entre 1 ano e menos de 5 anos', 'Entre 5 anos e menos de 10 anos',
            '10 anos ou mais', 'Nunca']),
    escala(['Há menos de 12 meses', 'Há 12 meses ou mais', 'Nunca']),
    escala(['Há menos de 6 meses', '6 a menos de 12 meses', 'Há 12 meses ou mais', 'Nunca']),
    escala(['Há menos de 10 anos', 'Entre 10 anos e menos de 20 anos', '20 anos ou mais', 'Nunca']),
    escala(['Data', 'Só ano', 'Há mais de 1 ano', 'Nunca']),

    # Frequências e quantidades
    escala(['Nunca', 'Menos de uma vez por semana', '1 a 3 vezes por semana', '4 a 6 vezes por semana',
            'Uma vez ou mais por dia']),
    escala(['Nunca', 'Menos de uma vez por semana', 'Algumas vezes por semana',
            'Todos os dias, apenas antes de se deitar', 'Todos os dias, de manhã e antes de se deitar',
            'Todos os dias, de manhã, após o almoço e antes de se deitar'],
           {'Todos os dias, apenas de manhã': 3}),
    escala(['10 a 29 minutos', '30 a 59 minutos', '1 hora a menos de 2 horas', '2 horas a menos de 3 horas',
            '3 horas ou mais']),
    escala(['Não executa nenhuma das tarefas indicadas',
            'Sentada ou em pé em atividades que envolvem um esforço físico ligeiro',
            'Em movimento ou em tarefas que exigem um esforço físico moderado',
            'Em trabalhos pesados ou fisicamente exigentes']),
    escala(['Menos de 10 horas', 'Entre 10 horas e menos de 20 horas', '20 horas ou mais']),

    # Tabaco e álcool
    escala(['Não fuma', 'Já fumou, mas atualmente não fuma', 'Fuma ocasionalmente', 'Fuma diariamente']),
    escala(['Não fumou', 'Ocasionalmente (menos de uma vez por mês)', 'Ocasionalmente (uma vez por mês)',
            'Ocasionalmente (alguns dias por mês)', 'Ocasionalmente (alguns dias por semana)', 'Diariamente']),
    escala(['Nunca', 'Não, mas já utilizou', 'Sim, ocasionalmente', 'Sim, diariamente']),
    escala(['Nunca ou quase nunca', 'Menos de uma vez por semana',
            'Pelo menos uma vez por semana (mas não diariamente)', 'Diariamente, menos de 1 hora por dia',
            'Diariamente, 1 hora ou mais por dia']),
    escala(['Nunca consumiu, ou só ocasionalmente para provar',
            'Não consumiu nos últimos 12 meses, por ter deixado de consumir álcool',
            'Menos de uma vez por mês', 'Uma vez por mês', '2 a 3 dias por mês', '1 a 2 dias por semana',
            '3 a 4 dias por semana', '5 a 6 dias por semana', 'Todos os dias ou quase todos os dias']),
    escala(['0 bebidas por dia', '1 bebida por dia', '2 bebidas por dia', '3 bebidas por dia',
            '4 a 5 bebidas por dia', '6 a 9 bebidas por dia', '10 a 15 bebidas por dia',
            '16 ou mais bebidas por dia']),
    escala(['Em nenhum dos dias', '1 dia', '2 dias', '3 dias', '4 dias']),

    # Opiniões e suporte social
    escala(['Totalmente em desacordo', 'Em desacordo', 'Mais ou menos em desacordo',
            'Nem de acordo nem em desacordo', 'Mais ou menos de acordo', 'De acordo', 'Totalmente de acordo']),
    escala(['Nenhuma', '1 ou 2', '3 a 5', '6 ou mais']),
    escala(['Nenhuma preocupação e interesse', 'Pouca preocupação e interesse',
            'Alguma preocupação e interesse', 'Muita preocupação e interesse'],
           {'Não consegue avaliar': -3}),
    escala(['Muito difícil', 'Difícil', 'Possível', 'Fácil', 'Muito fácil']),

    # Socioeconómicas
    escala(['Nenhum nível completado', 'Básico 1º e 2º ciclo', 'Básico 3º ciclo', 'Secundário',
            'Pós secundário', 'C.T.Superior Profissional', 'Bacharelato ou licenciatura 1º ciclo',
            'Licenciatura pré-bolonha ou mestrado 2º ciclo', 'Doutoramento']),
    escala(['1º quintil', '2º quintil', '3º quintil', '4º quintil', '5º quintil']),
    escala(['Áreas pouco povoadas', 'Áreas mediamente povoadas', 'Áreas densamente povoadas']),
]

NOMINAIS = ['REGION', 'NUTS2_AG', 'BIRTHPLACE_COD', 'BIRTHPLACEFATH_COD', 'BIRTHPLACEMOTH_COD',
            'CITIZEN_COD', 'MARSTALEGAL', 'MAINSTAT', 'JOBSTAT_COD', 'JOBISCO_COD', 'LOCNACE_COD',
            'HHTYPE', 'IN5', 'IN14', 'IN16', 'IN17', 'IN21', 'IN22', 'IN44', 'IN49', 'IN62', 'IC2']

print(f"{len(ESCALAS)} escalas ordinais/binárias definidas | {len(NOMINAIS)} colunas nominais")

44 escalas ordinais/binárias definidas | 22 colunas nominais


### 3.2 Aplicação da Codificação

A codificação é aplicada ao treino e ao teste em conjunto (sem o alvo), garantindo que ambos ficam com exatamente as mesmas colunas. Não há fuga de informação: apenas se usa a lista de respostas possíveis, nunca o alvo.

Nas variáveis numéricas, os valores em falta passam a **−1**, valor impossível nestas colunas (todas ≥ 0). O motivo da falta (não aplicável ou sem resposta) já está guardado nas colunas `_ESTADO`.

In [9]:
dados = pd.concat([train_df.drop(columns=TARGET), test_df])


def encontrar_escala(c):
    """Devolve a primeira escala que contém todas as respostas da coluna."""
    respostas = set(dados[c].unique()) - set(CODIGOS)
    for mapa in ESCALAS:
        if respostas <= set(mapa):
            return mapa
    return None


# 1) Ordinais e binárias
cols_ordinais = [c for c in cols_cat if c not in NOMINAIS]
sem_escala = [c for c in cols_ordinais if encontrar_escala(c) is None]
assert not sem_escala, f"Colunas sem escala definida: {sem_escala}"

for c in cols_ordinais:
    dados[c] = dados[c].map({**encontrar_escala(c), **CODIGOS_NUM})
assert dados[cols_ordinais].isna().sum().sum() == 0, "Há respostas que não estão em nenhuma escala"

# 2) Nominais -> one-hot
dados = pd.get_dummies(dados, columns=NOMINAIS, dtype=int)

# 3) Numéricas: valor em falta -> -1
dados[cols_num] = dados[cols_num].fillna(-1)

# Separar de novo treino e teste (pelo PID)
train_enc = dados.loc[train_df.index].copy()
train_enc[TARGET] = train_df[TARGET]
test_enc = dados.loc[test_df.index].copy()

print(f"Ordinais/binárias: {len(cols_ordinais)} | Nominais: {len(NOMINAIS)} -> "
      f"{dados.shape[1] - len(cols_ordinais) - len(cols_num)} colunas one-hot")
print(f"Treino: {train_enc.shape} | Teste: {test_enc.shape}")
print(f"Colunas não numéricas restantes: {dados.select_dtypes(exclude='number').shape[1]}")
print(f"Valores em falta restantes: {dados.isna().sum().sum()}")

Ordinais/binárias: 205 | Nominais: 22 -> 171 colunas one-hot
Treino: (10962, 410) | Teste: (3655, 409)
Colunas não numéricas restantes: 0
Valores em falta restantes: 0


## 4. Gravação dos Dados Codificados

Guardamos esta primeira versão dos dados codificados, que será usada no Notebook 3 para treinar um **modelo de referência** e fazer já uma submissão. As secções seguintes deste notebook (seleção e criação de features) produzirão versões melhoradas.

In [10]:
assert list(train_enc.columns.drop(TARGET)) == list(test_enc.columns)

train_enc.to_csv(os.path.join(DATA_DIR, 'train_nb02.csv'))
test_enc.to_csv(os.path.join(DATA_DIR, 'test_nb02.csv'))
print(f"Guardado: train_nb02.csv {train_enc.shape} | test_nb02.csv {test_enc.shape}")

Guardado: train_nb02.csv (10962, 410) | test_nb02.csv (3655, 409)


> **Conclusão:** as 205 variáveis ordinais e binárias foram codificadas numa única coluna cada, preservando a ordem das respostas; as 22 nominais deram origem a 171 colunas *one-hot*. O conjunto final tem 409 features, todas numéricas e sem valores em falta.